# A/B-тест: Cookie Cats

Датасет: [Mobile Games A/B Testing - Cookie Cats](https://www.kaggle.com/datasets/mursideyarkin/mobile-games-ab-testing-cookie-cats)

**Контекст.** Cookie Cats — мобильная игра-головоломка. В игре есть "ворота" (gate), которые заставляют игрока подождать или заплатить, прежде чем продолжить прохождение. Изначально ворота стояли на 30 уровне. Разработчики тестируют гипотезу: если сдвинуть ворота на 40 уровень, изменится ли удержание игроков (retention)?

- Группа `gate_30` — контроль (ворота на 30 уровне)
- Группа `gate_40` — тест (ворота на 40 уровне)

**Метрики:** `retention_1` (вернулся ли игрок через 1 день), `retention_7` (вернулся ли через 7 дней).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

sns.set_theme(style="whitegrid")

## 1. Загрузка данных

In [ ]:
df = pd.read_csv("../data/cookie_cats.csv")
df.head()

In [ ]:
df.info()
print("\nДубликаты по userid:", df["userid"].duplicated().sum())

## 2. Проверка дизайна эксперимента

Перед тем как считать статистику, убедимся, что группы сопоставимы по размеру (сплит примерно 50/50) и нет пересечений пользователей между группами.

In [ ]:
group_counts = df["version"].value_counts()
print(group_counts)

group_counts.plot(kind="bar", color=["#4C72B0", "#DD8452"])
plt.title("Размер групп")
plt.ylabel("Количество игроков")
plt.xticks(rotation=0)
plt.show()

## 3. Удержание по группам (retention_1 и retention_7)

In [ ]:
summary = df.groupby("version")[["retention_1", "retention_7"]].agg(["mean", "count"])
summary

In [ ]:
retention_means = df.groupby("version")[["retention_1", "retention_7"]].mean()

retention_means.plot(kind="bar", figsize=(7, 5))
plt.title("Retention rate по группам")
plt.ylabel("Доля вернувшихся игроков")
plt.xticks(rotation=0)
plt.legend(["1-day retention", "7-day retention"])
plt.show()

## 4. Формулировка гипотез

Для каждой метрики (`retention_1`, `retention_7`) проверяем:

- **H0**: доля вернувшихся игроков одинакова в группах `gate_30` и `gate_40` (перенос ворот не влияет на удержание).
- **H1**: доли различаются.

Используем двухвыборочный z-тест для пропорций (`statsmodels.stats.proportion.proportions_ztest`), уровень значимости `alpha = 0.05`.

In [ ]:
def run_proportions_test(df, metric, alpha=0.05):
    counts = df.groupby("version")[metric].sum()
    nobs = df.groupby("version")[metric].count()

    successes = np.array([counts["gate_30"], counts["gate_40"]])
    n_obs = np.array([nobs["gate_30"], nobs["gate_40"]])

    z_stat, p_value = proportions_ztest(successes, n_obs)

    ci_30 = proportion_confint(successes[0], n_obs[0], alpha=alpha)
    ci_40 = proportion_confint(successes[1], n_obs[1], alpha=alpha)

    print(f"--- {metric} ---")
    print(f"gate_30: {successes[0]/n_obs[0]:.4f}  95% CI: ({ci_30[0]:.4f}, {ci_30[1]:.4f})")
    print(f"gate_40: {successes[1]/n_obs[1]:.4f}  95% CI: ({ci_40[0]:.4f}, {ci_40[1]:.4f})")
    print(f"z-statistic: {z_stat:.4f}")
    print(f"p-value: {p_value:.4f}")
    if p_value < alpha:
        print(f"=> Различие статистически значимо (p < {alpha}), H0 отвергаем.")
    else:
        print(f"=> Значимого различия не обнаружено (p >= {alpha}), H0 не отвергаем.")
    print()

    return z_stat, p_value

In [ ]:
run_proportions_test(df, "retention_1")
run_proportions_test(df, "retention_7")

## 5. Bootstrap как альтернативная проверка

Дополнительно проверим результат для `retention_7` бутстрапом: многократно пересэмплируем каждую группу с возвращением, считаем разницу retention rate между группами и смотрим, насколько часто `gate_30` оказывается лучше `gate_40`.

In [ ]:
rng = np.random.default_rng(42)
n_bootstraps = 10000

gate_30 = df.loc[df["version"] == "gate_30", "retention_7"].to_numpy()
gate_40 = df.loc[df["version"] == "gate_40", "retention_7"].to_numpy()

boot_diff = np.empty(n_bootstraps)
for i in range(n_bootstraps):
    sample_30 = rng.choice(gate_30, size=len(gate_30), replace=True)
    sample_40 = rng.choice(gate_40, size=len(gate_40), replace=True)
    boot_diff[i] = sample_30.mean() - sample_40.mean()

prob_gate_30_better = (boot_diff > 0).mean()
ci_low, ci_high = np.percentile(boot_diff, [2.5, 97.5])

print(f"Доля бутстрап-выборок, где retention_7(gate_30) > retention_7(gate_40): {prob_gate_30_better:.4f}")
print(f"95% доверительный интервал разницы (gate_30 - gate_40): ({ci_low:.4f}, {ci_high:.4f})")

plt.figure(figsize=(7, 5))
sns.histplot(boot_diff, kde=True)
plt.axvline(0, color="red", linestyle="--", label="Нет разницы")
plt.title("Bootstrap-распределение разницы retention_7 (gate_30 - gate_40)")
plt.xlabel("Разница retention rate")
plt.legend()
plt.show()

## 6. Выводы

**Retention_1 (1 день):** `gate_30` ≈ 44.8%, `gate_40` ≈ 44.2%, p-value ≈ 0.07 — разница **статистически не значима** на уровне `alpha = 0.05`.

**Retention_7 (7 дней):** `gate_30` ≈ 19.0%, `gate_40` ≈ 18.2%, p-value заметно меньше 0.05 — разница **статистически значима**, `gate_30` даёт лучшее долгосрочное удержание.

**Bootstrap:** доля бутстрап-выборок, где `retention_7(gate_30) > retention_7(gate_40)`, около 96% — согласуется с z-тестом и не опирается на предпосылки нормальности.

**Рекомендация:** переносить ворота с 30 на 40 уровень не стоит. Значимой разницы в 1-дневном удержании нет, а по 7-дневному удержанию `gate_30` статистически значимо лучше — эксперимент говорит в пользу того, чтобы оставить ворота на текущем месте.